# LA Airbnb Listings — Data Cleaning

**Input:** `listings.csv` (45,533 rows x 25 cols, LA Airbnb snapshot)  
**Output:** `listings_clean.csv` — a cleaned `df` ready for EDA / unsupervised (KMeans on lat/lon, PCA on numerics) / supervised modeling of `price`.

**Scope of this notebook = cleaning only.** Feature engineering (e.g. host tenure days, target encoding, outlier winsorization) is left to the modeling teammates.

**Cleaning checklist (problems observed in raw data):**
1. `host_since` has *mixed* formats — some rows are `dd/mm/yy` strings, others are Excel serial numbers like `42368`.
2. Boolean columns (`host_is_superhost`, `instant_bookable`) are stored as `'t'`/`'f'` strings.
3. Target `price` has ~18% NaN (target rows must be dropped — can't impute a target).
4. `bathrooms` and `beds` have `0` as minimum (physically implausible — likely data errors).
5. `host_response_time`/`host_response_rate` NaN is *meaningful* (host never responded) — needs an indicator, not silent imputation.
6. `license` has 3 semantic tiers mixed into one column (NaN / "Exempt ..." / specific license numbers).
7. Text fields (`name`, `host_name`) may have stray whitespace / control chars.

Field groupings are detected *programmatically* so the notebook stays robust if the source schema shifts.

## 1. Load & initial profile

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

RAW_PATH   = 'listings.csv'
CLEAN_PATH = 'listings_clean.csv'

raw = pd.read_csv(RAW_PATH, low_memory=False)
print('raw shape:', raw.shape)
raw.head(3)

In [ ]:
# Quick profile: dtypes + nulls + uniques in one view
profile = pd.DataFrame({
    'dtype':   raw.dtypes.astype(str),
    'nulls':   raw.isna().sum(),
    'null_%':  (raw.isna().mean() * 100).round(1),
    'nunique': raw.nunique(dropna=True),
})
profile

In [ ]:
# Work on a copy so `raw` stays available for reference
df = raw.copy()
print('rows before cleaning:', len(df))

## 2. Parse `host_since` (mixed formats)

Raw column mixes two formats:
- `'dd/mm/yy'` strings like `'20/01/13'` (14,560 rows)
- Excel serial numbers like `'42368'` (30,971 rows) — these are days since Excel's 1900 epoch (origin `1899-12-30` accounts for Excel's 1900-leap-year bug)

Detection is by **string pattern**, not row count, so the code stays correct if the mix shifts.

In [ ]:
def parse_mixed_date(series: pd.Series) -> pd.Series:
    """Parse a series that mixes dd/mm/yy strings and Excel serial numbers into datetime."""
    s = series.astype('string').str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')

    serial_mask = s.str.fullmatch(r'\d+', na=False)
    slash_mask  = s.str.contains('/', na=False)

    if serial_mask.any():
        out.loc[serial_mask] = pd.to_datetime(
            s.loc[serial_mask].astype(int), origin='1899-12-30', unit='D'
        )
    if slash_mask.any():
        out.loc[slash_mask] = pd.to_datetime(
            s.loc[slash_mask], format='%d/%m/%y', errors='coerce'
        )
    return out

df['host_since'] = parse_mixed_date(df['host_since'])

print('host_since dtype :', df['host_since'].dtype)
print('host_since range :', df['host_since'].min(), '->', df['host_since'].max())
print('host_since nulls :', df['host_since'].isna().sum(), '(expect ~2 — original NaN rows)')

## 3. Boolean columns (`t`/`f` -> nullable boolean)

Detect programmatically: any `object` column whose non-null values are a subset of `{'t','f'}` is a t/f boolean. This avoids hardcoding a list.

In [ ]:
def detect_tf_columns(frame: pd.DataFrame) -> list:
    tf = set(['t', 'f'])
    cols = []
    for c in frame.select_dtypes(include='object').columns:
        vals = set(frame[c].dropna().unique())
        if vals and vals.issubset(tf):
            cols.append(c)
    return cols

tf_cols = detect_tf_columns(df)
print('t/f columns detected:', tf_cols)

# map t/f -> True/False while preserving NaN via pandas nullable BooleanDtype
for c in tf_cols:
    df[c] = df[c].map({'t': True, 'f': False}).astype('boolean')

df[tf_cols].dtypes

## 4. Clean target `price`

- **Drop rows with NaN `price`** — a regression target can't be imputed without leaking information.
- **Drop rows with `price <= 0`** — not a valid nightly rate. (In this dataset the current min is 5, so this is a defensive check.)
- **Do NOT clip high-side outliers here.** Whether $56,425/night is a real luxury listing or a data error is a modeling decision (winsorize? log transform? robust model?) — that belongs to the modeling teammates. We just flag it.

In [ ]:
before = len(df)
df = df[df['price'].notna() & (df['price'] > 0)].copy()
print(f'dropped {before - len(df)} rows with NaN or non-positive price -> {len(df)} rows remain')

print('\nprice distribution after cleaning:')
print(df['price'].describe(percentiles=[.01, .25, .5, .75, .95, .99, .999]).round(2))

# Report extreme values for the modeling team's awareness (no clipping here).
high_cutoff = df['price'].quantile(0.999)
print(f'\nrows above 99.9th percentile (price > {high_cutoff:.0f}): {(df["price"] > high_cutoff).sum()}')

## 5. Zero-value anomalies in `bathrooms` / `beds`

- `bathrooms == 0` is physically implausible for a rental -> treat as missing (NaN). Do **not** impute here.
- `beds == 0` is implausible for a nightly rental -> NaN.
- `bedrooms == 0` is **kept as-is** — it legitimately means a studio.

In [ ]:
implausible_zero_cols = ['bathrooms', 'beds']
for c in implausible_zero_cols:
    n_zero = (df[c] == 0).sum()
    df.loc[df[c] == 0, c] = np.nan
    print(f'{c}: {n_zero} zero values -> NaN')

df[['bathrooms', 'bedrooms', 'beds']].describe()

## 6. Host response columns — missingness has meaning

`host_response_time` and `host_response_rate` are both NaN for the same ~10,088 rows — these are hosts who have never replied to a guest message. Silently imputing (e.g. with median) would hide that signal.

Strategy: **keep the NaNs** but add a `host_never_responded` boolean indicator so downstream modeling can decide how to use it.

In [ ]:
rate_na = df['host_response_rate'].isna()
time_na = df['host_response_time'].isna()

# Sanity check: the two should overlap heavily
print('rate NaN             :', rate_na.sum())
print('time NaN             :', time_na.sum())
print('both NaN (aligned)   :', (rate_na & time_na).sum())

df['host_never_responded'] = (rate_na & time_na).astype('boolean')
print('\nhost_never_responded counts:')
print(df['host_never_responded'].value_counts(dropna=False))

## 7. Normalize `license` into a 3-level status

Raw `license` packs 3 concepts into one column:
- NaN -> no license on file (`'none'`)
- Starts with `'Exempt'` -> exempt from LA registration (`'exempt'`)
- Any other value (e.g. `'HSR19-004485'`, `'166412'`) -> a specific registered license (`'registered'`)

The specific license number itself carries no predictive signal, so we replace `license` with a clean categorical `license_status`.

In [ ]:
def classify_license(v):
    if pd.isna(v):
        return 'none'
    s = str(v).strip()
    if s.lower().startswith('exempt'):
        return 'exempt'
    return 'registered'

df['license_status'] = df['license'].map(classify_license).astype('category')
df = df.drop(columns=['license'])

print(df['license_status'].value_counts(dropna=False))

## 8. Text hygiene (`name`, `host_name`, other strings)

- Strip leading/trailing whitespace.
- Collapse internal repeated whitespace.
- Convert empty strings to NaN.

Applied programmatically to all remaining `object` columns.

In [ ]:
object_cols = df.select_dtypes(include='object').columns.tolist()
print('object columns to normalize:', object_cols)

for c in object_cols:
    s = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    s = s.replace({'': pd.NA})
    df[c] = s

## 9. Final consistency checks

Assertions that the cleaned `df` is ready for the downstream pipeline.

In [ ]:
assert df['price'].notna().all(),            'price has NaN'
assert (df['price'] > 0).all(),              'price has non-positive values'
assert df['id'].is_unique,                   'duplicate listing ids'
assert df['latitude'].between(33, 35).all(), 'lat outside LA bounds'
assert df['longitude'].between(-119, -117).all(), 'lon outside LA bounds'
assert pd.api.types.is_datetime64_any_dtype(df['host_since']), 'host_since not datetime'

# Show the new profile side-by-side with raw
final_profile = pd.DataFrame({
    'dtype':   df.dtypes.astype(str),
    'nulls':   df.isna().sum(),
    'null_%':  (df.isna().mean() * 100).round(1),
    'nunique': df.nunique(dropna=True),
})
final_profile

In [ ]:
# One-line summary of what cleaning changed
print(f'raw rows      : {len(raw):>6}')
print(f'clean rows    : {len(df):>6}  (-{len(raw)-len(df)} rows with NaN or non-positive price)')
print(f'raw columns   : {raw.shape[1]:>6}')
print(f'clean columns : {df.shape[1]:>6}  (license -> license_status, +host_never_responded)')

## 10. Export cleaned dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f'saved -> {CLEAN_PATH}  ({len(df)} rows, {df.shape[1]} cols)')